# 07 — Optimisation du système RAG avec MLFlow

**Objectif** : tester toutes les combinaisons de paramètres du pipeline RAG via un vrai pipeline end-to-end sans passer par l'API.

Chaque run MLFlow correspond à une configuration complète :

| Paramètre | Valeurs testées | Impact |
|---|---|---|
| `chunk_max_tokens` | 128, 256, 512 | Granularité du découpage |
| `chunk_overlap_tokens` | 10, 20, 40 | Continuité entre chunks |
| `retrieval_k` | 3, 5, 7 | Nombre de docs retournés |
| `system_prompt` | simple, concis, structuré, hyper-structuré | Style de réponse |

**Stratégie** : optimisation séquentielle (pas de grid search complet) pour limiter le nombre de runs :
1. Fixer les paramètres de chunking → trouver le meilleur `k`
2. Fixer `k` → trouver le meilleur `system_prompt`
3. Fixer `k` + `prompt` → trouver la meilleure config de chunking

> **Rate limiting** : juge `mistral-medium-latest` à 0.3 req/s. Avec 8 questions × 5 métriques × ~4 appels ≈ 160 appels/run ≈ 9 min/run. Avec 11 runs au total (3 valeurs de `k` + 4 prompts + 3 chunkings + 1 run final "optimal"), compter **~1h30 à 1h45 pour l'ensemble du notebook** — à lancer avec de la marge avant la soutenance, pas juste avant.

## 0. Configuration

In [1]:
import json
import os
import warnings

from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_mistralai import ChatMistralAI
import mlflow
import pandas as pd
from ragas import RunConfig, evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    AnswerRelevancy,
    ContextEntityRecall,
    ContextPrecision,
    ContextRecall,
    Faithfulness,
)

from api.infra.faiss.store import FAISSClient
from api.infra.mistral.embeddings import MistralEmbedder
from api.infra.preprocessing.chunker import TextChunker
from notebooks.config import (
    MISTRAL_API_KEY,
    MISTRAL_CHAT_MODEL,
    MISTRAL_EMBED_MODEL,
    MISTRAL_TEMPERATURE,
    PROJECT_ROOT,
)
from notebooks.mistral import mistral_chat
from notebooks.mlflow_utils import setup_experiment
from notebooks.prompts import SYSTEM_PROMPT as PRODUCTION_SYSTEM_PROMPT

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ragas 0.4.x still supports these imports/wrappers but warns about the future
# ragas.metrics.collections / llm_factory API — silenced here to keep the output readable.
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"ragas.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=r".*ragas.*")
warnings.filterwarnings(
    "ignore", category=DeprecationWarning, message=r".*Langchain(LLM|Embeddings)Wrapper.*"
)

# --- Juge RAGAS ---
MISTRAL_JUDGE_MODEL = "mistral-medium-latest"
RAGAS_RATE_LIMIT = 0.3  # req/s

rate_limiter = InMemoryRateLimiter(
    requests_per_second=RAGAS_RATE_LIMIT,
    check_every_n_seconds=1,
    max_bucket_size=1,
)

eval_llm = ChatMistralAI(
    model=MISTRAL_JUDGE_MODEL,
    mistral_api_key=MISTRAL_API_KEY,
    temperature=0.0,
    rate_limiter=rate_limiter,
)

# Embedder partagé pour le pipeline RAG (rebuild d'index)
embedder = MistralEmbedder(api_key=MISTRAL_API_KEY, model=MISTRAL_EMBED_MODEL)

# --- MLFlow ---
setup_experiment("rag-cultural-events")

EVAL_DIR = PROJECT_ROOT / "data/evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"LLM RAG        : {MISTRAL_CHAT_MODEL}")
print(f"LLM évaluateur : {MISTRAL_JUDGE_MODEL} ({RAGAS_RATE_LIMIT} req/s)")
print(f"Embedder       : {MISTRAL_EMBED_MODEL}")

/var/folders/6_/pkc98_l54jd53vtftrl1lb6w0000gn/T/ipykernel_71865/535926668.py:13: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
/var/folders/6_/pkc98_l54jd53vtftrl1lb6w0000gn/T/ipykernel_71865/535926668.py:13: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextEntityRecall
  from ragas.metrics import (
/var/folders/6_/pkc98_l54jd53vtftrl1lb6w0000gn/T/ipykernel_71865/535926668.py:13: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics 

LLM RAG        : mistral-small-latest
LLM évaluateur : mistral-medium-latest (0.3 req/s)
Embedder       : mistral-embed


## 1. Données source et questions de test

In [2]:
# Même corpus figé que le notebook 06 (200 événements versionnés dans tests/fixtures/documents.json,
# indépendant de l'index de production `models/faiss_index/` qui varie avec OPENAGENDA_MAX_RECORDS)
# — nécessaire pour que les scores de ce notebook restent comparables à ceux du notebook 06.
EVAL_CORPUS_PATH = PROJECT_ROOT / "tests/fixtures/documents.json"
parent_docs_raw: dict[str, str] = json.loads(EVAL_CORPUS_PATH.read_text(encoding="utf-8"))
print(f"{len(parent_docs_raw)} documents parents chargés depuis {EVAL_CORPUS_PATH}")

200 documents parents chargés depuis /Users/kilian/Documents/Repositories/OpenClassrooms/OC-Concevez-et-deployez-un-systeme-RAG/tests/fixtures/documents.json


In [3]:
# Sous-ensemble représentatif de 8 questions (extrait du jeu de test notebook 06)
EVAL_QUESTIONS = [
    {
        "question": "Où et à quelles dates se déroule la visite-atelier 'Prendre la mesure du temps' ?",
        "reference": "L'événement se déroule au Musée de la poste (34 Boulevard de Vaugirard, 75015 Paris) du 20 septembre 2025 à 12h30 au 21 septembre 2025 à 12h30.",
    },
    {
        "question": "Faut-il payer ou s'inscrire pour participer aux 'Nuits des étoiles au parc Montsouris' ?",
        "reference": "Non, les soirées sont gratuites et ouvertes à toutes et à tous sans inscription.",
    },
    {
        "question": "Quel équipement est fourni aux auditeurs lors de l'expérience sonore 'Les Sons de Voyager' ?",
        "reference": "Les auditeurs sont équipés de casques à conduction osseuse.",
    },
    {
        "question": "Qui animera la scène TWITCH lors du Festival MVP - Ma Vie Pro le 11 avril 2026 ?",
        "reference": "La scène TWITCH sera animée par le présentateur TV Samuel Etienne.",
    },
    {
        "question": "Pourquoi l'accès aux tribunes n'est-il pas permis lors de la visite de l'Église Saint-Vincent-de-Paul ?",
        "reference": "L'accès aux tribunes n'est pas permis en raison de la restauration des peintures à la cire de Flandrin lancée par la Mairie de Paris.",
    },
    {
        "question": "Quelles sont les quatre figures irlandaises majeures mises à l'honneur dans l'exposition 'Amitiés littéraires' au Centre Culturel Irlandais ?",
        "reference": "Les quatre figures sont James Joyce, Samuel Beckett, Francis Bacon et Oscar Wilde.",
    },
    {
        "question": "À quelle tranche d'âge le Forum 'SOS Rentrée' s'adresse-t-il spécifiquement ?",
        "reference": "Le forum s'adresse aux jeunes ayant entre 15 et 25 ans.",
    },
    {
        "question": "Dans quel lieu hybride se déroule le festival 'La Grande Semaine Végétale' à Paris ?",
        "reference": "Le festival se déroule à Plantation Paris, une ferme urbaine située au 37 Rue des Cheminots 75018 Paris.",
    },
]

print(f"{len(EVAL_QUESTIONS)} questions de test")

8 questions de test


## 2. Pipeline end-to-end

In [4]:
def build_faiss_index(
    parent_docs: dict[str, str],
    chunk_max_tokens: int,
    chunk_overlap_tokens: int,
) -> FAISSClient:
    """Re-chunke et re-embedde les parent docs, construit un index FAISS en mémoire."""
    chunker = TextChunker(
        chunk_max_tokens=chunk_max_tokens,
        chunk_overlap_tokens=chunk_overlap_tokens,
    )

    texts, metadatas = [], []
    for uid, doc_text in parent_docs.items():
        for i, fragment in enumerate(chunker.split_text(doc_text)):
            texts.append(fragment)
            metadatas.append({"uid": uid, "chunk_id": str(i)})

    print(
        f"  {len(parent_docs)} docs → {len(texts)} chunks (max={chunk_max_tokens}, overlap={chunk_overlap_tokens})"
    )

    embeddings = embedder.embed_texts(texts)

    client = FAISSClient(
        embed_texts_fn=embedder.embed_texts,
        embed_query_fn=embedder.embed_query,
    )
    client.build(texts, embeddings, metadatas, parent_docs)
    return client


def make_ask_fn(faiss_client: FAISSClient, system_prompt: str, retrieval_k: int):
    """Retourne une fonction ask() paramétrée."""

    def ask(question: str) -> dict:
        docs = faiss_client.search(question, k=retrieval_k)
        seen, sources = set(), []
        for doc in docs:
            uid = doc.metadata.get("uid")
            if uid and uid not in seen:
                seen.add(uid)
                text = faiss_client.get_parent_doc(uid)
                if text:
                    sources.append({"uid": uid, "text": text})
        if not sources:
            return {"answer": "Aucun événement pertinent trouvé.", "sources": []}
        context = "\n---\n".join(f"[{i + 1}] {s['text']}" for i, s in enumerate(sources))
        answer = mistral_chat(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Contexte :\n{context}\n\nQuestion :\n{question}"},
            ],
            api_key=MISTRAL_API_KEY,
            model=MISTRAL_CHAT_MODEL,
            temperature=MISTRAL_TEMPERATURE,
        )
        return {"answer": answer, "sources": sources}

    return ask


def run_ragas(ask_fn, questions: list[dict]) -> object:
    """Génère les réponses RAG puis évalue avec RAGAS."""
    records = []
    for item in questions:
        result = ask_fn(item["question"])
        records.append(
            {
                "user_input": item["question"],
                "retrieved_contexts": [s["text"] for s in result["sources"]],
                "response": result["answer"],
                "reference": item["reference"],
            }
        )

    evaluator_llm = LangchainLLMWrapper(eval_llm)
    from notebooks.mistral import MistralEmbeddings

    lc_embedder = MistralEmbeddings(api_key=MISTRAL_API_KEY, model=MISTRAL_EMBED_MODEL)
    evaluator_embeddings = LangchainEmbeddingsWrapper(lc_embedder)

    return evaluate(
        dataset=EvaluationDataset.from_list(records),
        metrics=[
            Faithfulness(llm=evaluator_llm),
            AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings, strictness=1),
            ContextRecall(llm=evaluator_llm),
            ContextPrecision(llm=evaluator_llm),
            ContextEntityRecall(llm=evaluator_llm),
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=RunConfig(timeout=600, max_workers=1, max_wait=120, max_retries=3),
    )


RAGAS_METRIC_NAMES = [
    "faithfulness",
    "answer_relevancy",
    "context_recall",
    "context_precision",
    "context_entity_recall",
]


def mean_scores(eval_result) -> dict[str, float]:
    """Extrait les scores moyens depuis un EvaluationResult RAGAS."""
    df = eval_result.to_pandas()
    return {name: float(df[name].mean()) for name in RAGAS_METRIC_NAMES}


def log_run(run_name: str, params: dict, eval_result) -> str:
    """Log un run MLFlow avec paramètres et métriques."""
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params)
        scores = mean_scores(eval_result)
        mlflow.log_metrics(scores)
        mean = sum(scores.values()) / len(scores)
        summary = " | ".join(f"{name}={value:.3f}" for name, value in scores.items())
        print(f"  [{run_name}] {summary} | mean={mean:.3f}")
        return run.info.run_id


print("Pipeline prêt.")

Pipeline prêt.


## 3. Définition des prompts

In [5]:
PROMPTS = {
    "simple": """Tu es un assistant spécialisé dans les événements culturels en Île-de-France.
Tu réponds uniquement à partir des événements fournis dans le contexte ci-dessous.

Règles :
- Réponds en français, de manière claire et concise (2 à 5 phrases).
- Ne crée jamais d'événement, de date, de lieu ou de prix non présent dans le contexte.
- Si le contexte ne permet pas de répondre, dis-le clairement.
- Cite les événements par leur titre et leur ville quand c'est pertinent.""",
    "concis": """Tu es un assistant sur les événements culturels en Île-de-France.
Réponds uniquement à partir du contexte fourni, en français, en 1 à 3 phrases.
Si l'information n'est pas dans le contexte, dis-le clairement.""",
    "structuré": """Tu es un assistant spécialisé dans les événements culturels en Île-de-France.
Tu réponds uniquement à partir des événements fournis dans le contexte.

Pour chaque réponse :
1. Cite le titre exact de l'événement concerné.
2. Donne l'information demandée de manière précise et directe.
3. Indique le lieu si pertinent.
4. Si l'information est absente du contexte, dis-le explicitement.

Réponse factuelle en 1 à 3 phrases maximum.""",
    "hyper-structuré": PRODUCTION_SYSTEM_PROMPT,
}

print(f"{len(PROMPTS)} variantes de system prompt : {list(PROMPTS.keys())}")

4 variantes de system prompt : ['simple', 'concis', 'structuré', 'hyper-structuré']


## 4. Expérience A — Optimisation de `retrieval_k`

On fixe chunking=baseline et prompt=`hyper-structuré` (prompt de production actuel), on fait varier `k`.

In [6]:
BASELINE_CHUNK = {"chunk_max_tokens": 256, "chunk_overlap_tokens": 20}
K_VALUES = [3, 5, 7]

print("=== Expérience A : retrieval_k ===")
print(
    f"Build index baseline (chunk={BASELINE_CHUNK['chunk_max_tokens']}, overlap={BASELINE_CHUNK['chunk_overlap_tokens']})..."
)
index_baseline = build_faiss_index(parent_docs_raw, **BASELINE_CHUNK)

k_scores = []
for k in K_VALUES:
    print(f"\nk={k} — évaluation RAGAS...")
    ask = make_ask_fn(index_baseline, PROMPTS["hyper-structuré"], retrieval_k=k)
    result = run_ragas(ask, EVAL_QUESTIONS)
    log_run(
        run_name=f"k={k}",
        params={
            **BASELINE_CHUNK,
            "retrieval_k": k,
            "system_prompt": "hyper-structuré",
            "chat_model": MISTRAL_CHAT_MODEL,
            "embed_model": MISTRAL_EMBED_MODEL,
            "n_questions": len(EVAL_QUESTIONS),
        },
        eval_result=result,
    )
    s = mean_scores(result)
    k_scores.append({"k": k, **s})

k_df = pd.DataFrame(k_scores).set_index("k")
k_df["mean"] = k_df.mean(axis=1)
display(k_df.round(3))

BEST_K = int(k_df["mean"].idxmax())
print(f"\n→ Meilleur k : {BEST_K}")

=== Expérience A : retrieval_k ===
Build index baseline (chunk=256, overlap=20)...
  200 docs → 417 chunks (max=256, overlap=20)


2026-07-07 19:24:51.085 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 32 texts
2026-07-07 19:24:51.580 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 2, 32 texts
2026-07-07 19:24:52.077 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 3, 32 texts
2026-07-07 19:24:52.616 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 4, 32 texts
2026-07-07 19:24:53.075 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 5, 32 texts
2026-07-07 19:24:53.527 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 6, 32 texts
2026-07-07 19:24:53.984 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 7, 32 texts
2026-07-07 19:24:54.770 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 8, 32 texts
2026-07-07 19:24:55.276 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 9, 32 texts
2026-07-07


k=3 — évaluation RAGAS...


2026-07-07 19:24:58.285 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:24:59.629 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:25:01.038 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:25:01.815 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:25:02.634 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:25:03.422 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:25:04.129 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:25:04.801 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

  [k=3] faithfulness=1.000 | answer_relevancy=0.938 | context_recall=1.000 | context_precision=0.979 | context_entity_recall=0.385 | mean=0.860

k=5 — évaluation RAGAS...


... (4 outputs left out)

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

  [k=7] faithfulness=1.000 | answer_relevancy=0.910 | context_recall=1.000 | context_precision=0.979 | context_entity_recall=0.404 | mean=0.859


,faithfulness,answer_relevancy,context_recall,context_precision,context_entity_recall,mean
k,,,,,,
3,1.0,0.938,1.0,0.979,0.385,0.860
5,1.0,0.936,1.0,0.979,0.385,0.860
7,1.0,0.910,1.0,0.979,0.404,0.859



→ Meilleur k : 3


## 5. Expérience B — Optimisation du `system_prompt`

On fixe chunking=baseline et k=best_k, on fait varier le prompt.

In [7]:
print(f"=== Expérience B : system_prompt (k={BEST_K}) ===")

prompt_scores = []
for prompt_name, prompt_text in PROMPTS.items():
    print(f"\nprompt={prompt_name} — évaluation RAGAS...")
    ask = make_ask_fn(index_baseline, prompt_text, retrieval_k=BEST_K)
    result = run_ragas(ask, EVAL_QUESTIONS)
    log_run(
        run_name=f"prompt={prompt_name}",
        params={
            **BASELINE_CHUNK,
            "retrieval_k": BEST_K,
            "system_prompt": prompt_name,
            "chat_model": MISTRAL_CHAT_MODEL,
            "embed_model": MISTRAL_EMBED_MODEL,
            "n_questions": len(EVAL_QUESTIONS),
        },
        eval_result=result,
    )
    s = mean_scores(result)
    prompt_scores.append({"prompt": prompt_name, **s})

prompt_df = pd.DataFrame(prompt_scores).set_index("prompt")
prompt_df["mean"] = prompt_df.mean(axis=1)
display(prompt_df.round(3))

BEST_PROMPT = prompt_df["mean"].idxmax()
print(f"\n→ Meilleur prompt : {BEST_PROMPT}")

=== Expérience B : system_prompt (k=3) ===

prompt=simple — évaluation RAGAS...


2026-07-07 19:42:08.923 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:42:09.669 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:42:10.903 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:42:11.780 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:42:12.605 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:42:13.431 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:42:14.406 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:42:15.561 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

  [prompt=simple] faithfulness=1.000 | answer_relevancy=0.941 | context_recall=1.000 | context_precision=0.979 | context_entity_recall=0.375 | mean=0.859

prompt=concis — évaluation RAGAS...


2026-07-07 19:46:32.642 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:46:34.517 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:46:37.352 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:46:39.139 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:46:39.874 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:46:41.104 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:46:45.459 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 19:46:46.576 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

... (6 outputs left out)

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

  [prompt=hyper-structuré] faithfulness=1.000 | answer_relevancy=0.933 | context_recall=1.000 | context_precision=0.979 | context_entity_recall=0.467 | mean=0.876


,faithfulness,answer_relevancy,context_recall,context_precision,context_entity_recall,mean
prompt,,,,,,
simple,1.000,0.941,1.0,0.979,0.375,0.859
concis,1.000,0.950,1.0,0.979,0.417,0.869
structuré,0.906,0.889,1.0,0.979,0.404,0.836
hyper-structuré,1.000,0.933,1.0,0.979,0.467,0.876



→ Meilleur prompt : hyper-structuré


## 6. Expérience C — Optimisation du chunking

On fixe k=best_k et prompt=best_prompt. On rebuild un nouvel index pour chaque config de chunking.

**Note** : chaque rebuild coûte ~N appels à `mistral-embed` (N = nombre de chunks). Avec ~200 events ça prend 30-60 secondes par config.

In [8]:
CHUNK_CONFIGS = [
    {"chunk_max_tokens": 128, "chunk_overlap_tokens": 10, "name": "small"},
    {"chunk_max_tokens": 256, "chunk_overlap_tokens": 20, "name": "baseline"},
    {"chunk_max_tokens": 512, "chunk_overlap_tokens": 40, "name": "medium"},
]

print(f"=== Expérience C : chunking (k={BEST_K}, prompt={BEST_PROMPT}) ===")

chunk_scores = []
for cfg in CHUNK_CONFIGS:
    name = cfg["name"]
    max_t = cfg["chunk_max_tokens"]
    overlap = cfg["chunk_overlap_tokens"]

    print(f"\nchunk={name} (max={max_t}, overlap={overlap}) — rebuild index...")
    index = build_faiss_index(
        parent_docs_raw, chunk_max_tokens=max_t, chunk_overlap_tokens=overlap
    )

    print("  évaluation RAGAS...")
    ask = make_ask_fn(index, PROMPTS[BEST_PROMPT], retrieval_k=BEST_K)
    result = run_ragas(ask, EVAL_QUESTIONS)
    log_run(
        run_name=f"chunk={name}",
        params={
            "chunk_max_tokens": max_t,
            "chunk_overlap_tokens": overlap,
            "retrieval_k": BEST_K,
            "system_prompt": BEST_PROMPT,
            "chat_model": MISTRAL_CHAT_MODEL,
            "embed_model": MISTRAL_EMBED_MODEL,
            "n_questions": len(EVAL_QUESTIONS),
        },
        eval_result=result,
    )
    s = mean_scores(result)
    chunk_scores.append({"chunk": name, "max_tokens": max_t, **s})

chunk_df = pd.DataFrame(chunk_scores).set_index("chunk")
chunk_df["mean"] = chunk_df[RAGAS_METRIC_NAMES].mean(axis=1)
display(chunk_df.round(3))

BEST_CHUNK = chunk_df["mean"].idxmax()
print(f"\n→ Meilleure config de chunking : {BEST_CHUNK}")

=== Expérience C : chunking (k=3, prompt=hyper-structuré) ===

chunk=small (max=128, overlap=10) — rebuild index...
  200 docs → 797 chunks (max=128, overlap=10)


2026-07-07 20:00:24.176 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 32 texts
2026-07-07 20:00:24.820 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 2, 32 texts
2026-07-07 20:00:25.276 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 3, 32 texts
2026-07-07 20:00:25.831 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 4, 32 texts
2026-07-07 20:00:26.451 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 5, 32 texts
2026-07-07 20:00:26.845 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 6, 32 texts
2026-07-07 20:00:27.285 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 7, 32 texts
2026-07-07 20:00:28.227 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 8, 32 texts
2026-07-07 20:00:28.854 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 9, 32 texts
2026-07-07

  évaluation RAGAS...


2026-07-07 20:00:41.431 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 20:00:42.243 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 20:00:42.998 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 20:00:43.855 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 20:00:44.621 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 20:00:45.850 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts
2026-07-07 20:00:46.670 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 1 texts


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

  [chunk=small] faithfulness=0.875 | answer_relevancy=0.830 | context_recall=0.875 | context_precision=0.875 | context_entity_recall=0.385 | mean=0.768

chunk=baseline (max=256, overlap=20) — rebuild index...
  200 docs → 417 chunks (max=256, overlap=20)


... (8 outputs left out)

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

  [chunk=medium] faithfulness=1.000 | answer_relevancy=0.931 | context_recall=1.000 | context_precision=0.979 | context_entity_recall=0.287 | mean=0.839


,max_tokens,faithfulness,answer_relevancy,context_recall,context_precision,context_entity_recall,mean
chunk,,,,,,,
small,128,0.875,0.830,0.875,0.875,0.385,0.768
baseline,256,1.000,0.940,1.000,0.979,0.435,0.871
medium,512,1.000,0.931,1.000,0.979,0.287,0.839



→ Meilleure config de chunking : baseline


## 7. Comparaison globale dans MLFlow

In [9]:
experiment = mlflow.get_experiment_by_name("rag-cultural-events")
if experiment:
    runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.faithfulness DESC"],
    )
    cols = {
        "tags.mlflow.runName": "run",
        "params.chunk_max_tokens": "chunk_tokens",
        "params.retrieval_k": "k",
        "params.system_prompt": "prompt",
        "metrics.faithfulness": "faithfulness",
        "metrics.answer_relevancy": "answer_relevancy",
        "metrics.context_recall": "context_recall",
    }
    available = {k: v for k, v in cols.items() if k in runs_df.columns}
    display(
        runs_df[list(available.keys())].rename(columns=available).round(3).reset_index(drop=True)
    )
else:
    print("Expérience MLFlow non trouvée — exécuter les cellules précédentes.")

,run,chunk_tokens,k,prompt,faithfulness,answer_relevancy,context_recall
0,chunk=medium,512,3,hyper-structuré,1.000,0.931,1.000
1,chunk=baseline,256,3,hyper-structuré,1.000,0.940,1.000
2,prompt=hyper-structuré,256,3,hyper-structuré,1.000,0.933,1.000
3,prompt=concis,256,3,concis,1.000,0.950,1.000
4,prompt=simple,256,3,simple,1.000,0.941,1.000
5,k=7,256,7,hyper-structuré,1.000,0.910,1.000
6,k=5,256,5,hyper-structuré,1.000,0.936,1.000
7,k=3,256,3,hyper-structuré,1.000,0.938,1.000
8,prompt=structuré,256,3,structuré,0.906,0.889,1.000
9,baseline,256,5,None,0.897,0.910,0.933


## 8. Configuration optimale — run final

In [10]:
best_cfg = next(c for c in CHUNK_CONFIGS if c["name"] == BEST_CHUNK)

print("Configuration optimale :")
print(f"  chunk_max_tokens     = {best_cfg['chunk_max_tokens']}")
print(f"  chunk_overlap_tokens = {best_cfg['chunk_overlap_tokens']}")
print(f"  retrieval_k          = {BEST_K}")
print(f"  system_prompt        = {BEST_PROMPT}")
print()

print("Rebuild de l'index optimal et évaluation finale...")
index_optimal = build_faiss_index(
    parent_docs_raw,
    chunk_max_tokens=best_cfg["chunk_max_tokens"],
    chunk_overlap_tokens=best_cfg["chunk_overlap_tokens"],
)
ask_optimal = make_ask_fn(index_optimal, PROMPTS[BEST_PROMPT], retrieval_k=BEST_K)
result_optimal = run_ragas(ask_optimal, EVAL_QUESTIONS)

scores_path = EVAL_DIR / "ragas_optimal.parquet"
result_optimal.to_pandas().to_parquet(scores_path, index=False)

log_run(
    run_name="optimal",
    params={
        "chunk_max_tokens": best_cfg["chunk_max_tokens"],
        "chunk_overlap_tokens": best_cfg["chunk_overlap_tokens"],
        "retrieval_k": BEST_K,
        "system_prompt": BEST_PROMPT,
        "chat_model": MISTRAL_CHAT_MODEL,
        "embed_model": MISTRAL_EMBED_MODEL,
        "n_questions": len(EVAL_QUESTIONS),
    },
    eval_result=result_optimal,
)

print(f"\n✓ Run optimal enregistré → {scores_path}")
print("  Visualiser tous les runs : mlflow ui")

Configuration optimale :
  chunk_max_tokens     = 256
  chunk_overlap_tokens = 20
  retrieval_k          = 3
  system_prompt        = hyper-structuré

Rebuild de l'index optimal et évaluation finale...
  200 docs → 417 chunks (max=256, overlap=20)


2026-07-07 20:32:06.510 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 1, 32 texts
2026-07-07 20:32:06.964 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 2, 32 texts
2026-07-07 20:32:07.759 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 3, 32 texts
2026-07-07 20:32:08.307 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 4, 32 texts
2026-07-07 20:32:09.452 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 5, 32 texts
2026-07-07 20:32:09.900 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 6, 32 texts
2026-07-07 20:32:10.488 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 7, 32 texts
2026-07-07 20:32:11.614 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 8, 32 texts
2026-07-07 20:32:12.171 | DEBUG    | api.infra.mistral.embeddings:embed_texts:49 - Embedded batch 9, 32 texts
2026-07-07

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

  [optimal] faithfulness=1.000 | answer_relevancy=0.924 | context_recall=1.000 | context_precision=0.979 | context_entity_recall=0.379 | mean=0.856

✓ Run optimal enregistré → /Users/kilian/Documents/Repositories/OpenClassrooms/OC-Concevez-et-deployez-un-systeme-RAG/data/evaluation/ragas_optimal.parquet
  Visualiser tous les runs : mlflow ui


## Bilan

### Résultats du run du 07/07 (corpus corrigé — `tests/fixtures/documents.json`, 200 événements)

Les résultats **s'inversent** par rapport à un run antérieur qui chargeait par erreur
`models/faiss_index/parent_docs.json` (l'index de production) au lieu du corpus d'évaluation figé
utilisé par le notebook 06. Ce bug de source de données est corrigé (section "1. Données source").

### Lecture des métriques

| Métrique | Ce qu'elle mesure | Impactée par |
|---|---|---|
| **Faithfulness** | Réponse ancrée dans le contexte (pas d'hallucination) | `system_prompt` |
| **AnswerRelevancy** | Réponse adresse bien la question | `system_prompt`, `k` |
| **ContextRecall** | Retriever a retrouvé le bon document | `chunk_max_tokens`, `k` |
| **ContextPrecision** | Peu de bruit dans le contexte retrouvé | `chunk_max_tokens`, `k` |
| **ContextEntityRecall** | Entités de la référence couvertes par le contexte | `chunk_max_tokens`, `k` |

**Expérience A — `retrieval_k`** (chunk=256/20, prompt=`hyper-structuré` fixés) :

| k | Faithfulness | AnswerRel. | ContextRecall | ContextPrec. | ContextEntRec. | Mean |
|---|---|---|---|---|---|---|
| 3 | 1.000 | 0.938 | 1.000 | 0.979 | 0.385 | **0.860** |
| 5 | 1.000 | 0.936 | 1.000 | 0.979 | 0.385 | 0.860 |
| 7 | 1.000 | 0.910 | 1.000 | 0.979 | 0.404 | 0.859 |

→ **k=3/5/7 sont statistiquement indistinguables** (0.859-0.860) — pas de signal exploitable sur
cet échantillon de 8 questions, l'écart est de l'ordre du bruit inter-run déjà documenté (notebook
06). `k=3` retenu par défaut (valeur la plus économe en tokens de contexte à qualité égale), ce qui
correspond à la valeur déjà en production (`RETRIEVAL_K=3`).

**Expérience B — `system_prompt`** (chunk=256/20, k=3 fixés) :

| Prompt | Faithfulness | AnswerRel. | ContextRecall | ContextPrec. | ContextEntRec. | Mean |
|---|---|---|---|---|---|---|
| simple | 1.000 | 0.941 | 1.000 | 0.979 | 0.375 | 0.859 |
| concis | 1.000 | 0.950 | 1.000 | 0.979 | 0.417 | 0.869 |
| structuré | 0.906 | 0.889 | 1.000 | 0.979 | 0.404 | 0.836 |
| **hyper-structuré (production)** | 1.000 | 0.933 | 1.000 | 0.979 | 0.467 | **0.876** |

→ **`hyper-structuré` (le prompt de production) arrive en tête** des 4 candidats sur ce corpus
corrigé — à l'inverse du run précédent (biaisé par le mauvais corpus) qui l'avait classé dernier.
C'est aussi le seul des 4 candidats à inclure des instructions explicites anti-prompt-injection
(voir notebook 06, section 5bis) — les 3 autres n'en ont aucune. Ce résultat a motivé la
restauration de ce prompt en production (`src/api/modules/rag/prompts.py`), qui avait été
temporairement remplacé par le candidat "simple" sur la base de l'ancien run erroné.

**Expérience C — chunking** (k=3, prompt=`hyper-structuré` fixés) :

| Chunk | max_tokens | Faithfulness | AnswerRel. | ContextRecall | ContextPrec. | ContextEntRec. | Mean |
|---|---|---|---|---|---|---|---|
| small | 128 | 0.875 | 0.830 | 0.875 | 0.875 | 0.385 | 0.768 |
| **baseline** | 256 | 1.000 | 0.940 | 1.000 | 0.979 | 0.435 | **0.871** |
| medium | 512 | 1.000 | 0.931 | 1.000 | 0.979 | 0.287 | 0.839 |

→ **`baseline` (256/20) domine sur toutes les métriques** (égal ou meilleur que `medium` partout,
et nettement meilleur sur `context_entity_recall` : 0.435 vs 0.287). `small` (128/10) est
clairement inférieur sur toutes les métriques de retrieval. Ce résultat inverse aussi la conclusion
du run précédent (qui donnait `medium` à égalité avec `baseline`, sur la base du mauvais corpus) et
a motivé le retour de `CHUNK_MAX_TOKENS`/`CHUNK_OVERLAP_TOKENS` à 256/20 en production (`.env`,
`.env.example`, `src/api/common/config.py`) — un `/rebuild` est nécessaire pour que l'index de
production reflète ce changement.

### Configuration optimale retenue par ce sweep

`chunk_max_tokens=256`, `chunk_overlap_tokens=20`, `retrieval_k=3`, `system_prompt="hyper-structuré"`
→ faithfulness=1.000, answer_relevancy=0.924, context_recall=1.000, context_precision=0.979,
context_entity_recall=0.379, **mean=0.856**.

Cette configuration optimale **correspond exactement à la configuration de production après les
changements ci-dessus** — contrairement au run précédent, dont la configuration "optimale"
recommandait de s'écarter de la production alors en place. Le score final (0.856) est cohérent
avec le score individuel de l'expérience B pour `hyper-structuré` (0.876) — pas de grosse surprise
d'interaction entre paramètres cette fois, contrairement à ce qu'on observait sur le run précédent.

### Limites de cette expérimentation

- Échantillon volontairement réduit à 8 questions nominales (sans piège) pour limiter la
  consommation d'API — **pas comparable** aux scores du notebook 06 (15 questions, dont 5 pièges).
  Ces 8 questions restent codées en dur dans ce notebook (extrait manuel du jeu du notebook 06) —
  elles peuvent diverger si `data/evaluation/test_questions.json` est modifié sans mise à jour
  manuelle correspondante ici.
- `context_entity_recall` reste la métrique la plus bruitée sur cet échantillon (0.375-0.467 selon
  le prompt, 0.287-0.435 selon le chunking) — les écarts sur cette seule métrique ne suffiraient
  pas à eux seuls à justifier un changement de configuration, mais ici `baseline` et
  `hyper-structuré` gagnent aussi sur les autres métriques (ou sont à égalité), pas seulement sur
  celle-ci.
- Cette recherche d'hyperparamètres reste une trace exploratoire de la méthodologie suivie ; elle a
  cette fois directement motivé deux changements de configuration en production (chunking et
  system prompt), après correction du bug de corpus qui faussait le run précédent.